# EasyMagpieTTS — offline two-stage synthesis

Runs EasyMagpie LM → SpectralCodec-BWE-22kHz in a single `AsyncOmni` engine (no external codec service)
and writes a `.wav` file.

First [convert the NeMo checkpoint](../../../tools/easymagpie_vllm_omni/README.md#convert-a-nemo-checkpoint) and
[set up the standalone serving environment](../../../tools/easymagpie_vllm_omni/README.md#setup-the-serving-environment).
Launch this notebook from the Speech repository using that environment's Jupyter kernel, then set `MODEL_DIR`
below to the converted model containing the bundled `codec/` directory.

In [ ]:
import os
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

import json
from pathlib import Path

import vllm_plugin_easymagpie_omni
vllm_plugin_easymagpie_omni.register()

from vllm import SamplingParams
from vllm.sampling_params import RequestOutputKind
from vllm_omni import AsyncOmni
from transformers import AutoTokenizer

from easymagpie_vllm_omni.audio_output import extract_audio_from_stage_output
from easymagpie_vllm_omni.config import EasyMagpieOmniArch
from easymagpie_vllm_omni.easymagpie import EasyMagpieTTSForConditionalGeneration

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "tools" / "easymagpie_vllm_omni").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within a Speech repository checkout")


REPO_ROOT = find_repo_root()
EASYMAGPIE_ROOT = REPO_ROOT / "tools" / "easymagpie_vllm_omni"
MODEL_DIR = str(EASYMAGPIE_ROOT / "converted_model")
DEPLOY_CONFIG = str(EASYMAGPIE_ROOT / "deploy" / "easymagpie.yaml")
TEXT = "Hello, welcome to the text-to-speech demo."
SPEAKER_ID = "eng"
OUT_WAV = "out.wav"
MAX_NEW_TOKENS = 1024

In [ ]:
def load_meta(model_dir: str, speaker_id: str):
    config = json.loads((Path(model_dir) / "config.json").read_text())
    arch = EasyMagpieOmniArch.from_hf_config(type("Cfg", (), config))
    tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    prompt_len = EasyMagpieTTSForConditionalGeneration.get_prompt_len(
        speaker_id, model_dir, tokenize=lambda t: tokenizer.encode(t)
    )
    stop_token_id = EasyMagpieTTSForConditionalGeneration.audio_eos_stop_token_id(type("Cfg", (), config))
    return {
        "prompt_len": int(prompt_len),
        "stop_token_id": int(stop_token_id),
    }


def build_prompt(text: str, prompt_len: int, speaker_id: str) -> dict:
    return {
        "prompt_token_ids": [0] * prompt_len,
        "additional_information": {
            "context_text": "[EN]",
            "text": text,
            "temperature": 0.7,
            "top_k": 80,
            "speaker_id": speaker_id,
        },
    }


async def synthesize_one(omni, text: str, meta: dict, speaker_id: str, max_new_tokens: int):
    lm_sp = SamplingParams(
        temperature=0.0,
        max_tokens=max_new_tokens,
        detokenize=False,
        ignore_eos=False,
        stop_token_ids=[meta["stop_token_id"]],
        output_kind=RequestOutputKind.DELTA,
    )
    code2wav_sp = SamplingParams(temperature=0.0, max_tokens=max_new_tokens, detokenize=True)
    prompt = build_prompt(text, meta["prompt_len"], speaker_id)
    audio = None
    gen = omni.generate(
        prompt,
        sampling_params_list=[lm_sp, code2wav_sp],
        request_id=f"easymp-{abs(hash(text)) & 0xFFFF:x}",
    )
    async for stage_output in gen:
        extracted = extract_audio_from_stage_output(stage_output)
        if extracted is not None:
            audio = extracted
    return audio

In [ ]:
import asyncio
import soundfile as sf
from IPython.display import Audio, display

meta = load_meta(MODEL_DIR, SPEAKER_ID)
print(f"model={MODEL_DIR}  prompt_len={meta['prompt_len']}  stop={meta['stop_token_id']}")

omni = AsyncOmni(model=MODEL_DIR, deploy_config=DEPLOY_CONFIG, log_stats=False)
try:
    wav, sr = await synthesize_one(omni, TEXT, meta, SPEAKER_ID, MAX_NEW_TOKENS)
    sf.write(OUT_WAV, wav, sr)
    print(f"Wrote {OUT_WAV} ({len(wav)/sr:.2f}s @ {sr} Hz)")
    display(Audio(wav, rate=sr))
finally:
    omni.shutdown()